# Sequence View of `generatedata` Datasets

Any dataset that has an `x_y_index` (i.e. a pixel/label split) can be reshaped
into a time-series by calling `load_data_as_sequence(name, step_size=...)`.

- `step_size` controls how many feature values form one timestep.
- `seq_len` is computed automatically as `x_y_index // step_size`.
- No special datasets need to be generated — the reshaping happens at load time.

In [34]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display
import torch
from torch import nn

from generatedata.load_data import load_data, load_data_as_sequence, data_names

---
## 2  API Reference — `load_data_as_sequence()`

```python
load_data_as_sequence(
    name: str,
    step_size: int,
    local: bool = False,
    data_dir: Path | str | None = None,
    label_every_step: bool = True,
) -> tuple[np.ndarray, np.ndarray]
```

**Key points**

| Detail | Value |
|---|---|
| `step_size` | Runtime parameter — never stored in `info.json` |
| `seq_len` | Computed as `x_y_index // step_size` |
| Error if `x_y_index` missing | `ValueError: Dataset '...' has no x_y_index metadata` |
| Error if not divisible | `ValueError: x_y_index (...) is not evenly divisible by step_size (...)` |

**Return shapes**

| `label_every_step` | `X_seq.shape` | `labels.shape` |
|---|---|---|
| `True` (default) | `(num_points, seq_len, step_size + label_dim)` | `(num_points, label_dim)` |
| `False` | `(num_points, seq_len, step_size)` | `(num_points, label_dim)` |

---
## 3  Discovering Compatible Datasets

Datasets with `x_y_index` in their metadata can be reshaped into sequences.
Datasets without it (e.g. geometric manifolds) cannot.

In [35]:
all_names = data_names(local=True)
print(f"All datasets ({len(all_names)}):")
for n in sorted(all_names):
    try:
        info = load_data(n, local=True)["info"]
        if "x_y_index" in info:
            print(f"  {n:50s}  x_y_index={info['x_y_index']}  y_size={info['y_size']}")
        else:
            print(f"  {n:50s}  (no x_y_index — cannot load as sequence)")
    except Exception:
        print(f"  {n:50s}  (error loading)")

All datasets (14):
  EMlocalization                                      x_y_index=160  y_size=1
  LunarLander                                         x_y_index=404  y_size=4
  MNIST                                               x_y_index=784  y_size=10
  MNIST1D                                             x_y_index=40  y_size=10
  MNIST1D_seq1                                        x_y_index=40  y_size=10
  MNIST1Dcustom_scale0.4_maxtrans48_corrnoise0.25_iidnoise0.02_shear0.75  x_y_index=40  y_size=10
  MNIST_custom_degrees0_0_translate0_0_scale1_1       x_y_index=2500  y_size=10
  MNIST_seq1                                          x_y_index=784  y_size=10
  MassSpec                                            x_y_index=921  y_size=512
  circle                                              (no x_y_index — cannot load as sequence)
  manifold                                            (no x_y_index — cannot load as sequence)
  pca_line                                            (no x_y_i

---
## 4  Choosing a Dataset and `step_size`

The relationship between the parameters:

```
seq_len = x_y_index // step_size
```

A **smaller** `step_size` → longer sequence, fewer features per step.  
A **larger** `step_size` → shorter sequence, more features per step.

`step_size` must divide `x_y_index` exactly — you can inspect valid values with
`[d for d in range(1, x_y_index + 1) if x_y_index % d == 0]`.

Edit `DATASET` and `STEP_SIZE` below and re-run to explore any compatible dataset.

In [36]:
# ── Pick your dataset and step_size ─────────────────────────────────────────
DATASET   = "MNIST"   # any dataset with x_y_index
STEP_SIZE = 1         # must divide x_y_index evenly

X_seq, labels = load_data_as_sequence(DATASET, step_size=STEP_SIZE, local=True,
                                       label_every_step=True)
info = load_data(DATASET, local=True)["info"]

print(f"Dataset      : {DATASET}")
print(f"step_size    : {STEP_SIZE}")
print(f"seq_len      : {X_seq.shape[1]}  (= x_y_index / step_size = {info['x_y_index']} / {STEP_SIZE})")
print(f"X_seq shape  : {X_seq.shape}   (num_points, seq_len, step_size + label_dim)")
print(f"labels shape : {labels.shape}")

Dataset      : MNIST
step_size    : 1
seq_len      : 784  (= x_y_index / step_size = 784 / 1)
X_seq shape  : (1000, 784, 11)   (num_points, seq_len, step_size + label_dim)
labels shape : (1000, 10)


---
## 5  Visualisations

These plots work for any compatible dataset. Change `DATASET` / `STEP_SIZE` in section 4
and re-run this section to visualise a different dataset.

### 5.1  Time-series signal — first few samples

In [37]:
# Load pixels-only for visualisation (no label concatenation needed)
X_vis, labels_vis = load_data_as_sequence(DATASET, step_size=STEP_SIZE, local=True,
                                           label_every_step=False)
seq_len = X_vis.shape[1]

sample_indices = [0, 1, 2]
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=[
        f"Sample {i}  |  label: {int(np.argmax(labels_vis[i]))}"
        for i in sample_indices
    ],
)

for col, idx in enumerate(sample_indices, start=1):
    # Flatten all step channels into a single 1-D signal
    signal = X_vis[idx].reshape(-1)
    fig.add_trace(
        go.Scatter(
            y=signal,
            mode='lines' if len(signal) > 80 else 'lines+markers',
            line=dict(color='steelblue', width=0.9),
            showlegend=False,
        ),
        row=1, col=col,
    )
    fig.update_xaxes(title_text="Feature index", row=1, col=col)
    fig.update_yaxes(title_text="Value", row=1, col=col)

fig.update_layout(
    title=f"{DATASET}  —  pixel signal (step_size={STEP_SIZE}, seq_len={seq_len})",
    height=350,
)
fig.show()

### 5.2  Sequence heatmap — timestep × feature channel

Shows sample 0 as a 2-D heatmap: timesteps on the x-axis, feature channels within each
step on the y-axis.

In [38]:
sample_idx = 0
seq = X_vis[sample_idx]          # (seq_len, step_size)
label = int(np.argmax(labels_vis[sample_idx]))

fig = go.Figure(
    go.Heatmap(
        z=seq.T,                 # (step_size, seq_len)
        colorscale='RdBu',
        zmid=0,
        colorbar=dict(title="Value"),
        hovertemplate="timestep %{x}<br>channel %{y}<br>value %{z:.4f}<extra></extra>",
    )
)
fig.update_layout(
    title=f"{DATASET}  —  heatmap  |  sample {sample_idx}, label {label}  |  step_size={STEP_SIZE}",
    xaxis_title="Timestep",
    yaxis_title=f"Feature channel (0–{STEP_SIZE - 1})",
    height=300,
)
fig.show()

# ── Optional: 28×28 image reconstruction for MNIST-like datasets ─────────────
if info.get("x_y_index") == 784:
    img = X_vis[sample_idx].reshape(-1)[:784].reshape(28, 28)
    fig2 = go.Figure(
        go.Heatmap(z=img, colorscale='gray', showscale=False,
                   hovertemplate="row %{y}  col %{x}<br>val %{z:.3f}<extra></extra>")
    )
    fig2.update_layout(
        title=f"{DATASET}  —  28×28 image reconstruction  |  sample {sample_idx}, digit {label}",
        yaxis_autorange='reversed',
        xaxis_showticklabels=False,
        yaxis_showticklabels=False,
        height=350, width=370,
    )
    fig2.show()

### 5.3  Label distribution

In [39]:
label_ids = np.argmax(labels_vis, axis=1)
unique, counts = np.unique(label_ids, return_counts=True)

fig = go.Figure(
    go.Bar(
        x=unique,
        y=counts,
        marker_color='steelblue',
        text=counts,
        textposition='outside',
    )
)
fig.update_layout(
    title=f"Label distribution — {DATASET}",
    xaxis=dict(title="Class (argmax of one-hot label)", tickmode='linear', dtick=1),
    yaxis_title="Count",
    height=320,
)
fig.show()

---
## 6  Error Handling

`load_data_as_sequence` raises a descriptive `ValueError` in two situations:
- `step_size` does not evenly divide `x_y_index`
- The dataset has no `x_y_index` metadata at all

In [40]:
# Invalid step_size — does not divide x_y_index=784
try:
    load_data_as_sequence("MNIST", step_size=3, local=True)
except ValueError as e:
    print(f"ValueError: {e}")

# Dataset without x_y_index
try:
    load_data_as_sequence("circle", step_size=1, local=True)
except ValueError as e:
    print(f"ValueError: {e}")

ValueError: x_y_index (784) is not evenly divisible by step_size (3).
ValueError: Dataset 'circle' has no x_y_index metadata. Cannot reshape as sequence.


---
## 7  Interactive Dataset Explorer — Sequence Builder

This explorer lets you:

- **Choose any compatible dataset** (any dataset with `x_y_index`)
- **Select `pixels_per_step`** — any divisor of `x_y_index` — which determines the
  timestep width and therefore the total sequence length
- **Build the sequence entry by entry** using the *Up to step* slider — drag it from 1
  to `seq_len` to watch the sequence grow

Three panels update in sync:

| Panel | Content |
|---|---|
| **Left** | Revealed pixel signal (all features decoded so far, as a 1-D trace) |
| **Centre** | Heatmap of revealed timesteps × features-per-step |
| **Right** | For MNIST (784 px): 28×28 image with unrevealed pixels masked; for other datasets: one-hot label bar |

Changing `pixels_per_step` re-segments the same flat feature array — no re-download needed.

In [41]:
# ---------------------------------------------------------------------------
# Raw-data cache — load flat feature arrays once per dataset
# ---------------------------------------------------------------------------
_raw_cache = {}

def get_raw_data(name):
    """Return (pixels, labels, info) for the named dataset.

    pixels : (num_points, x_y_index)   float32
    labels : (num_points, label_dim)   float32
    info   : dict from info.json
    """
    if name not in _raw_cache:
        data   = load_data(name, local=True)
        info   = data["info"]
        tgt    = data["target"]
        idx    = info["x_y_index"]
        pixels = tgt.iloc[:, :idx].to_numpy(dtype=np.float32)
        labels = tgt.iloc[:, idx:].to_numpy(dtype=np.float32)
        _raw_cache[name] = (pixels, labels, info)
    return _raw_cache[name]


def divisors(n):
    """Return all positive divisors of n in ascending order."""
    return [d for d in range(1, n + 1) if n % d == 0]


# Detect all datasets that have a pixel/label split (x_y_index present)
_base_names = []
for _n in sorted(data_names(local=True)):
    try:
        _inf = load_data(_n, local=True)["info"]
        if "x_y_index" in _inf:
            _base_names.append(_n)
    except Exception:
        pass

print("Datasets available in the sequence builder:")
for _n in _base_names:
    _pix, _, _inf = get_raw_data(_n)
    print(f"  {_n:50s}  x_y_index={_pix.shape[1]}  label_dim={_inf['y_size']}")

Datasets available in the sequence builder:
  EMlocalization                                      x_y_index=160  label_dim=1
  LunarLander                                         x_y_index=404  label_dim=4
  MNIST                                               x_y_index=784  label_dim=10
  MNIST1D                                             x_y_index=40  label_dim=10
  MNIST1D_seq1                                        x_y_index=40  label_dim=10
  MNIST1Dcustom_scale0.4_maxtrans48_corrnoise0.25_iidnoise0.02_shear0.75  x_y_index=40  label_dim=10
  MNIST_custom_degrees0_0_translate0_0_scale1_1       x_y_index=2500  label_dim=10
  MNIST_seq1                                          x_y_index=784  label_dim=10
  MassSpec                                            x_y_index=921  label_dim=512
  regression_circle                                   x_y_index=1  label_dim=1
  regression_line                                     x_y_index=1  label_dim=1


In [42]:
# ---------------------------------------------------------------------------
# FigureWidgets — created once, mutated in-place
# ---------------------------------------------------------------------------
fw_sig = go.FigureWidget(layout=go.Layout(
    title="Revealed feature signal",
    xaxis_title="Feature index",
    yaxis_title="Value",
    height=340,
    margin=dict(t=55, b=50, l=55, r=20),
))
fw_sig.add_scatter(mode='lines', line=dict(width=0.8, color='steelblue'), name='signal')

fw_map = go.FigureWidget(layout=go.Layout(
    title="Sequence heatmap",
    xaxis_title="Timestep",
    yaxis_title="Feature within step",
    height=340,
    margin=dict(t=55, b=50, l=60, r=20),
))
fw_map.add_heatmap(colorscale='RdBu', zmid=0, showscale=True)

fw_img = go.FigureWidget(layout=go.Layout(
    title="Revealed image / label",
    height=340,
    margin=dict(t=55, b=50, l=50, r=20),
))
fw_img.add_heatmap(colorscale='gray', showscale=False)


# ---------------------------------------------------------------------------
# Update function
# ---------------------------------------------------------------------------
def update_builder(base_name, sample_idx, pixels_per_step, up_to_step, show_image):
    pixels, labels, info = get_raw_data(base_name)
    total_pixels = pixels.shape[1]
    label_dim    = info["y_size"]
    n_points     = info["num_points"]

    # Guard: pixels_per_step must divide total_pixels
    if total_pixels % pixels_per_step != 0:
        valid = [d for d in divisors(total_pixels) if d <= pixels_per_step]
        pixels_per_step = valid[-1] if valid else 1

    seq_len    = total_pixels // pixels_per_step
    up_to_step = max(1, min(up_to_step, seq_len))
    sample_idx = min(sample_idx, n_points - 1)

    pixel_flat = pixels[sample_idx]          # (total_pixels,)
    lbl        = labels[sample_idx]          # (label_dim,)
    digit      = int(np.argmax(lbl))

    # Reshape the full flat array into (seq_len, pixels_per_step)
    seq_full  = pixel_flat.reshape(seq_len, pixels_per_step)

    # Partial: only the first up_to_step timesteps
    n_revealed  = up_to_step * pixels_per_step
    partial_seq = seq_full[:up_to_step]      # (up_to_step, pixels_per_step)
    pct         = int(100 * up_to_step / seq_len)

    # ── left: revealed feature signal ────────────────────────────────────────
    with fw_sig.batch_update():
        fw_sig.data[0].x = list(range(n_revealed))
        fw_sig.data[0].y = pixel_flat[:n_revealed].tolist()
        fw_sig.data[0].mode = 'lines' if n_revealed > 80 else 'lines+markers'
        fw_sig.layout.title.text = (
            f"{base_name}  sample {sample_idx}  |  "
            f"{n_revealed}/{total_pixels} features revealed ({pct}%)  |  label: {digit}"
        )

    # ── centre: heatmap of revealed timesteps ────────────────────────────────
    with fw_map.batch_update():
        fw_map.data[0].z = partial_seq.T.tolist() if up_to_step > 0 else [[]]
        fw_map.layout.title.text = (
            f"Step {up_to_step}/{seq_len}  "
            f"({pixels_per_step} feat/step  →  seq_len={seq_len})"
        )
        fw_map.layout.xaxis.title.text = f"Timestep (0–{up_to_step - 1})"
        fw_map.layout.yaxis.title.text = f"Feature within step (0–{pixels_per_step - 1})"

    # ── right: image (perfect-square datasets) or label bar ─────────────────
    side = int(np.sqrt(total_pixels))
    is_square = (side * side == total_pixels and show_image)
    if is_square:
        img_flat = np.full(total_pixels, np.nan, dtype=np.float32)
        img_flat[:n_revealed] = pixel_flat[:n_revealed]
        img = img_flat.reshape(side, side)

        if not isinstance(fw_img.data[0], go.Heatmap):
            fw_img.data = []
            fw_img.add_heatmap(colorscale='gray', showscale=False,
                               hovertemplate="row %{y}  col %{x}<br>val %{z:.3f}<extra></extra>")
        with fw_img.batch_update():
            fw_img.data[0].z = img.tolist()
            fw_img.layout.title.text = f"{side}×{side} image — step {up_to_step}/{seq_len}  |  digit: {digit}"
            fw_img.layout.yaxis.autorange  = 'reversed'
            fw_img.layout.xaxis.showticklabels = False
            fw_img.layout.yaxis.showticklabels = False
    else:
        bar_colors = ['tomato' if i == digit else 'steelblue' for i in range(label_dim)]
        if not isinstance(fw_img.data[0], go.Bar):
            fw_img.data = []
            fw_img.add_bar()
        with fw_img.batch_update():
            fw_img.data[0].x = list(range(label_dim))
            fw_img.data[0].y = lbl.tolist()
            fw_img.data[0].marker.color = bar_colors
            fw_img.layout.title.text  = f"Label vector  |  predicted class: {digit}"
            fw_img.layout.xaxis.title.text = "Class"
            fw_img.layout.yaxis.title.text = "One-hot value"
            fw_img.layout.yaxis.autorange  = True
            fw_img.layout.xaxis.showticklabels = True


# ---------------------------------------------------------------------------
# Widgets
# ---------------------------------------------------------------------------
_init_name   = _base_names[0] if _base_names else "MNIST"
_init_pixels, _, _init_info = get_raw_data(_init_name)
_init_total  = _init_pixels.shape[1]
_init_divs   = divisors(_init_total)
_init_seqlen = _init_total   # start with pixels_per_step=1

wb_dataset = widgets.Dropdown(
    options=_base_names,
    value=_init_name,
    description='Dataset:',
    layout=widgets.Layout(width='300px'),
)

wb_sample = widgets.IntSlider(
    value=0, min=0, max=_init_info['num_points'] - 1, step=1,
    description='Sample:',
    continuous_update=False,
    layout=widgets.Layout(width='380px'),
    style={'description_width': '70px'},
)

wb_pps = widgets.SelectionSlider(
    options=_init_divs,
    value=1,
    description='feat / step:',
    continuous_update=False,
    readout=True,
    layout=widgets.Layout(width='380px'),
    style={'description_width': '80px'},
)

wb_step = widgets.IntSlider(
    value=_init_seqlen, min=1, max=_init_seqlen, step=1,
    description='Up to step:',
    continuous_update=False,
    layout=widgets.Layout(width='380px'),
    style={'description_width': '80px'},
)

wb_show_img = widgets.Checkbox(
    description='Show image (perfect-square datasets)',
)


def _on_wb_dataset_change(change):
    pix, _, inf = get_raw_data(change['new'])
    total = pix.shape[1]
    divs  = divisors(total)
    wb_pps.options  = divs
    wb_pps.value    = 1
    wb_sample.max   = inf['num_points'] - 1
    wb_sample.value = 0
    wb_step.max     = total
    wb_step.value   = total


def _on_wb_pps_change(change):
    pix, _, _ = get_raw_data(wb_dataset.value)
    total     = pix.shape[1]
    pps       = change['new']
    seq_len   = total // pps
    frac = wb_step.value / wb_step.max if wb_step.max > 0 else 1.0
    wb_step.max   = seq_len
    wb_step.value = max(1, min(round(frac * seq_len), seq_len))


wb_dataset.observe(_on_wb_dataset_change, names='value')
wb_pps.observe(_on_wb_pps_change, names='value')


# ── layout ───────────────────────────────────────────────────────────────────
ui_builder = widgets.VBox([
    widgets.HTML(
        "<b>Sequence Builder</b> — drag <i>Up to step</i> to reveal the sequence "
        "one timestep at a time; change <i>feat/step</i> to re-segment."
    ),
    widgets.HBox([wb_dataset, wb_sample]),
    widgets.HBox([wb_pps,    wb_step]),
    widgets.HBox([wb_show_img]),
])

out_builder = widgets.interactive_output(
    update_builder,
    dict(
        base_name       = wb_dataset,
        sample_idx      = wb_sample,
        pixels_per_step = wb_pps,
        up_to_step      = wb_step,
        show_image      = wb_show_img,
    ),
)

panels_builder = widgets.HBox(
    [fw_sig, fw_map, fw_img],
    layout=widgets.Layout(width='100%'),
)

display(ui_builder, panels_builder, out_builder)

# Initial render
update_builder(
    wb_dataset.value, wb_sample.value,
    wb_pps.value, wb_step.value, wb_show_img.value,
)

    'data': [{'line': {'color': 'steelblue', 'width': 0.8},
              'mode'…

Output()

## 8  Training a Simple LSTM Classifier

With the sequence API in place we can feed data directly into a recurrent neural
network.  This section trains a small LSTM on any of the available datasets and
plots the resulting loss / accuracy curves plus a confusion matrix.

**How it works**

| `pixels_per_step` | `seq_len = x_y_index // pixels_per_step` |
|-------------------|------------------------------------------|
| 1 | 784 (one pixel per timestep — very long) |
| 28 | 28 (one row of a 28×28 image per step) |
| 784 | 1 (whole image as a single step; LSTM = MLP) |

* `load_data_as_sequence(..., label_every_step=False)` returns `y` of shape
  `(N, n_classes)` — only the final label, not one per timestep.
* The LSTM reads the sequence and its **last hidden state** is passed to a
  linear classification head.

In [53]:
# ---------------------------------------------------------------------------
# 8.1  Training configuration — edit here to change the experiment
# ---------------------------------------------------------------------------
LSTM_DATASET       = "MNIST"   # any name returned by _base_names
LSTM_PIXELS_PER_STEP = 2         # must divide x_y_index of the chosen dataset
LSTM_HIDDEN_SIZE   = 64
LSTM_NUM_LAYERS    = 1
LSTM_NUM_EPOCHS    = 200
LSTM_BATCH_SIZE    = 64
LSTM_LEARNING_RATE = 1e-3
LSTM_VAL_FRAC      = 0.2
LSTM_DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Dataset          : {LSTM_DATASET}")
print(f"Pixels per step  : {LSTM_PIXELS_PER_STEP}")
print(f"Device           : {LSTM_DEVICE}")

Dataset          : MNIST
Pixels per step  : 2
Device           : cuda


In [54]:
# ---------------------------------------------------------------------------
# 8.2  Load data and build DataLoaders
# ---------------------------------------------------------------------------
X_seq, y_flat = load_data_as_sequence(
    name            = LSTM_DATASET,
    step_size       = LSTM_PIXELS_PER_STEP,
    local           = False,
    label_every_step= False,
)

# X_seq : (N, seq_len, pixels_per_step)   float32
# y_flat: (N, n_classes)                  one-hot → convert to class indices
seq_len_lstm = X_seq.shape[1]
n_classes    = y_flat.shape[1]
y_idx        = np.argmax(y_flat, axis=1).astype(np.int64)  # (N,)

print(f"X_seq shape : {X_seq.shape}  (N, seq_len={seq_len_lstm}, feat={LSTM_PIXELS_PER_STEP})")
print(f"y_idx shape : {y_idx.shape}  classes: {n_classes}")

# Train / validation split
N          = X_seq.shape[0]
n_val      = int(N * LSTM_VAL_FRAC)
n_train    = N - n_val
rng        = np.random.default_rng(42)
perm       = rng.permutation(N)
train_idx, val_idx = perm[:n_train], perm[n_train:]

X_tr = torch.tensor(X_seq[train_idx], dtype=torch.float32)
y_tr = torch.tensor(y_idx[train_idx], dtype=torch.long)
X_va = torch.tensor(X_seq[val_idx],   dtype=torch.float32)
y_va = torch.tensor(y_idx[val_idx],   dtype=torch.long)

train_ds = torch.utils.data.TensorDataset(X_tr, y_tr)
val_ds   = torch.utils.data.TensorDataset(X_va, y_va)
train_dl = torch.utils.data.DataLoader(train_ds, batch_size=LSTM_BATCH_SIZE, shuffle=True)
val_dl   = torch.utils.data.DataLoader(val_ds,   batch_size=LSTM_BATCH_SIZE, shuffle=False)

print(f"Train: {n_train}  |  Val: {n_val}")

X_seq shape : (1000, 392, 2)  (N, seq_len=392, feat=2)
y_idx shape : (1000,)  classes: 10
Train: 800  |  Val: 200


In [55]:
# ---------------------------------------------------------------------------
# 8.3  Model definition
# ---------------------------------------------------------------------------
class LSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super().__init__()
        self.lstm   = nn.LSTM(
            input_size  = input_size,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            batch_first = True,
        )
        self.head = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        # x: (batch, seq_len, input_size)
        out, _ = self.lstm(x)        # out: (batch, seq_len, hidden_size)
        last    = out[:, -1, :]      # final timestep hidden state
        return self.head(last)       # (batch, num_classes)


model     = LSTMClassifier(LSTM_PIXELS_PER_STEP, LSTM_HIDDEN_SIZE,
                           LSTM_NUM_LAYERS, n_classes).to(LSTM_DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LSTM_LEARNING_RATE)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters : {total_params:,}")
print(model)

Parameters : 18,058
LSTMClassifier(
  (lstm): LSTM(2, 64, batch_first=True)
  (head): Linear(in_features=64, out_features=10, bias=True)
)


In [56]:
# ---------------------------------------------------------------------------
# 8.4  Training loop
# ---------------------------------------------------------------------------
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}


def _run_epoch(loader, training=True):
    if training:
        model.train()
    else:
        model.eval()

    total_loss, correct, n = 0.0, 0, 0
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for xb, yb in loader:
            xb, yb = xb.to(LSTM_DEVICE), yb.to(LSTM_DEVICE)
            logits = model(xb)
            loss   = criterion(logits, yb)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(yb)
            correct    += (logits.argmax(1) == yb).sum().item()
            n          += len(yb)
    return total_loss / n, correct / n


print(f"{'Epoch':>5}  {'Train loss':>10}  {'Val loss':>10}  "
      f"{'Train acc':>10}  {'Val acc':>10}")
print("-" * 55)

for epoch in range(1, LSTM_NUM_EPOCHS + 1):
    tr_loss, tr_acc = _run_epoch(train_dl, training=True)
    va_loss, va_acc = _run_epoch(val_dl,   training=False)
    history["train_loss"].append(tr_loss)
    history["val_loss"].append(va_loss)
    history["train_acc"].append(tr_acc)
    history["val_acc"].append(va_acc)
    if epoch % 5 == 0 or epoch == 1:
        print(f"{epoch:>5}  {tr_loss:>10.4f}  {va_loss:>10.4f}  "
              f"{tr_acc:>10.3f}  {va_acc:>10.3f}")

print("-" * 55)
print(f"Final val accuracy: {history['val_acc'][-1]:.3f}")

Epoch  Train loss    Val loss   Train acc     Val acc
-------------------------------------------------------
    1      2.3044      2.2984       0.099       0.130
    5      2.2969      2.3033       0.110       0.075
   10      2.2974      2.3031       0.114       0.115
   15      2.2970      2.3008       0.114       0.115
   20      2.2966      2.3025       0.113       0.075
   25      2.2968      2.3019       0.114       0.115
   30      2.2965      2.3024       0.113       0.075
   35      2.2967      2.3029       0.114       0.115
   40      2.2966      2.3022       0.114       0.115
   45      2.2974      2.3003       0.114       0.115
   50      2.2968      2.3017       0.105       0.075
   55      2.2964      2.3019       0.114       0.115
   60      2.2869      2.2444       0.140       0.225
   65      2.2482      2.2489       0.160       0.185
   70      2.1912      2.1971       0.193       0.225
   75      2.1912      2.1938       0.190       0.225
   80      2.2046      2.1

In [57]:
# ---------------------------------------------------------------------------
# 8.5  Visualise training curves and confusion matrix
# ---------------------------------------------------------------------------
epochs_range = list(range(1, LSTM_NUM_EPOCHS + 1))

# ── training curves ──────────────────────────────────────────────────────────
fig_hist = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Loss", "Accuracy"),
)
for col, (metric, label) in enumerate(
    [("loss", "Loss"), ("acc", "Accuracy")], start=1
):
    fig_hist.add_scatter(
        x=epochs_range, y=history[f"train_{metric}"],
        mode='lines+markers', name=f"train {label}",
        row=1, col=col,
    )
    fig_hist.add_scatter(
        x=epochs_range, y=history[f"val_{metric}"],
        mode='lines+markers', name=f"val {label}",
        line=dict(dash='dash'),
        row=1, col=col,
    )

fig_hist.update_layout(
    title_text=(
        f"LSTM training — {LSTM_DATASET}  "
        f"(pps={LSTM_PIXELS_PER_STEP}, seq_len={seq_len_lstm})"
    ),
    height=380,
)
fig_hist.show()

# ── confusion matrix ─────────────────────────────────────────────────────────
model.eval()
all_preds, all_true = [], []
with torch.no_grad():
    for xb, yb in val_dl:
        xb = xb.to(LSTM_DEVICE)
        preds = model(xb).argmax(1).cpu().numpy()
        all_preds.append(preds)
        all_true.append(yb.numpy())

all_preds = np.concatenate(all_preds)
all_true  = np.concatenate(all_true)

cm = np.zeros((n_classes, n_classes), dtype=int)
for t, p in zip(all_true, all_preds):
    cm[t, p] += 1

# Normalise rows to get recall-like heatmap
cm_norm = cm.astype(float)
row_sums = cm_norm.sum(axis=1, keepdims=True)
cm_norm  = np.where(row_sums > 0, cm_norm / row_sums, 0.0)

fig_cm = go.Figure(go.Heatmap(
    z            = cm_norm.tolist(),
    colorscale   = 'Blues',
    text         = cm.tolist(),
    texttemplate = "%{text}",
    showscale    = True,
))
fig_cm.update_layout(
    title_text  = f"Confusion matrix (normalised by true class) — val set",
    xaxis_title = "Predicted class",
    yaxis_title = "True class",
    yaxis       = dict(autorange='reversed'),
    height      = 480,
)
fig_cm.show()